# Prediksi Nilai Berikutnya Tiap Parameter QC (Univariat) — Produk MT-620

Notebook ini memprediksi **nilai berikutnya dari setiap parameter** berdasarkan **data historis parameter itu sendiri** (pendekatan *univariate time-series*). Setiap parameter **dilatih dan di-split secara TERPISAH** (model sendiri-sendiri) sehingga mudah dipahami dan dibandingkan.

**13 parameter yang dianalisis (nama lab -> kolom data):**

| No | Nama Parameter (Lab) | Kolom Dataset |
|----|----------------------|---------------|
| 1  | Transmission         | `Transmission` |
| 2  | Appearance           | `APE` |
| 3  | Tin Content          | `Tin` |
| 4  | RI @ 25°C            | `RI` |
| 5  | SG @ 25°C            | `SG` |
| 6  | Acid Value           | `Acid` |
| 7  | Sulfur               | `Sulfur` |
| 8  | Water Content        | `Water` |
| 9  | Monomethyltin        | `Mono` |
| 10 | Yellowish Index      | `Yellow` |
| 11 | 2-EH                 | `EH` |
| 12 | Viscosity            | `Visco` |
| 13 | Pt-Co                | `PT` |

> **Catatan penting.** Parameter **Appearance (APE)** bersifat **kategorikal/konstan** (mis. selalu "Clear"), sehingga tidak dapat diprediksi sebagai angka. Untuk parameter ini ditampilkan **modus** (nilai paling sering) sebagai "prediksi". 12 parameter lain diprediksi secara numerik.

## Metodologi Singkat

Untuk tiap parameter numerik dilakukan:
1. **Pembentukan fitur lag** (lag1..lag5 + rata-rata bergerak) sebagai input — dasar pemodelan deret waktu (Box & Jenkins, 1970).
2. **Split berurutan waktu 80:20** (tanpa pengacakan, agar tidak terjadi kebocoran masa depan) — (Tashman, 2000; Hyndman & Athanasopoulos, 2021).
3. **Pelatihan XGBoost Regressor** (Chen & Guestrin, 2016).
4. **Evaluasi** MAE, RMSE, MAPE, R² + pembanding **baseline naif** (Hyndman & Koehler, 2006).
5. **Prediksi nilai berikutnya** + **interval 95%**.

## 1. Import Library

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
RANDOM_STATE = 42

: 

## 2. Membaca & Mengurutkan Data (kronologis)

In [ ]:
file_path = r"Data_Chemical_2026.04-16-MT620.xlsx"   # sesuaikan path bila perlu
df = pd.read_excel(file_path).drop_duplicates()
df['Tanggal'] = pd.to_datetime(df['Tanggal'], errors='coerce')
df = df.sort_values(['Tanggal','No']).reset_index(drop=True)
print('Jumlah baris:', len(df), '| rentang tanggal:', df['Tanggal'].min(), '-', df['Tanggal'].max())
df.head()

## 3. Pemetaan 13 Parameter

In [ ]:
# nama tampilan (lab) -> kolom dataset
param_map = {
    'Transmission':'Transmission', 'Appearance':'APE', 'Tin Content':'Tin',
    'RI @ 25C':'RI', 'SG @ 25C':'SG', 'Acid Value':'Acid', 'Sulfur':'Sulfur',
    'Water Content':'Water', 'Monomethyltin':'Mono', 'Yellowish Index':'Yellow',
    '2-EH':'EH', 'Viscosity':'Visco', 'Pt-Co':'PT',
}
# parameter numerik (semua kecuali Appearance)
numeric_params = {k:v for k,v in param_map.items() if v!='APE'}
for col in numeric_params.values():
    df[col] = pd.to_numeric(df[col], errors='coerce')
print('Total parameter:', len(param_map), '| numerik:', len(numeric_params), '| kategorikal: 1 (Appearance)')

## 4. Penanganan Parameter Appearance (kategorikal)

In [ ]:
ape = df['APE'].astype(str).str.strip()
ape = ape[ape.str.lower()!='nan']
if ape.nunique() <= 1:
    print(f'Appearance KONSTAN -> nilai: "{ape.mode().iloc[0]}"  (tidak perlu/ tidak bisa diprediksi numerik)')
else:
    print('Distribusi Appearance:'); print(ape.value_counts())
    print(f'\nPrediksi (modus) = "{ape.mode().iloc[0]}"')

## 5. Fungsi Pembentuk Fitur Lag
Fitur dibentuk dari nilai-nilai sebelumnya (lag) + rata-rata bergerak. Pemilihan lag mengacu pada analisis autokorelasi deret waktu (Box & Jenkins, 1970).

In [ ]:
LAGS=[1,2,3,4,5]; ROLL=5
def buat_fitur(series):
    s = series.dropna().reset_index(drop=True)
    d = pd.DataFrame({'y': s})
    for k in LAGS: d[f'lag{k}'] = s.shift(k)
    d['rolling_mean'] = s.shift(1).rolling(ROLL).mean()
    d = d.dropna().reset_index(drop=True)
    fitur = [f'lag{k}' for k in LAGS] + ['rolling_mean']
    return d, fitur, s

## 6. Fungsi Proses SATU Parameter (split + latih + evaluasi + prediksi)
Setiap parameter mendapat **split, model, evaluasi, dan prediksi sendiri**.

In [ ]:
def proses_parameter(nama, series, plot=True):
    d, fitur, s = buat_fitur(series)
    X = d[fitur].values; y = d['y'].values
    n = len(d); cut = int(n*0.8)
    Xtr,Xte,ytr,yte = X[:cut],X[cut:],y[:cut],y[cut:]

    # XGBRegressor yang lebih konservatif untuk mencegah overfitting (max_depth diturunkan, L1/L2 ditingkatkan)
    model = XGBRegressor(
        n_estimators=100,          # Mengurangi jumlah pohon
        learning_rate=0.03,       # Kecepatan belajar lebih konservatif
        max_depth=2,              # Membatasi kedalaman pohon ke 2 untuk kestabilan
        subsample=0.8,            # Subsampling baris
        colsample_bytree=0.8,     # Subsampling kolom
        reg_lambda=10.0,          # Regularisasi L2
        reg_alpha=2.0,            # Regularisasi L1
        random_state=RANDOM_STATE, 
        verbosity=0
    )
    model.fit(Xtr, ytr)
    yp = model.predict(Xte)
    naive = Xte[:,0]                       # baseline naif = nilai sebelumnya (lag1)

    rmse = lambda a,b: float(np.sqrt(mean_squared_error(a,b)))
    # Menghindari pembagian dengan nol/dekat-nol dengan np.maximum
    mape = lambda a,b: float(np.mean(np.abs((np.array(a,float)-b)/np.maximum(np.abs(np.array(a,float)), 1e-5)))*100)
    
    # Interval prediksi 95% dihitung dari residual data uji (validation/out-of-sample) 
    # agar lebih jujur dan tidak overconfident seperti training residual
    resid_val = yte - yp
    sigma = float(np.std(resid_val, ddof=1)) if len(resid_val)>1 else float(np.std(resid_val))
    
    last = np.array([[s.iloc[-k] for k in LAGS] + [s.iloc[-ROLL:].mean()]])
    pred = float(model.predict(last)[0])

    # Clip batas bawah ke 0 jika data aktualnya tidak pernah negatif (misal parameter QC kimia)
    is_non_negative = float(np.min(series)) >= 0
    lo = max(0.0, pred - 1.96*sigma) if is_non_negative else (pred - 1.96*sigma)
    hi = pred + 1.96*sigma

    hasil = dict(Parameter=nama, n=n,
        R2=round(r2_score(yte,yp),3), MAE=round(mean_absolute_error(yte,yp),4),
        RMSE=round(rmse(yte,yp),4), MAPE=round(mape(yte,yp),2),
        MAE_baseline=round(mean_absolute_error(yte,naive),4),
        Prediksi_berikutnya=round(pred,4),
        Batas_bawah_95=round(lo,4), Batas_atas_95=round(hi,4))
    hasil['Menang_vs_baseline'] = 'YA' if hasil['MAE']<hasil['MAE_baseline'] else 'tidak'

    if plot:
        plt.figure(figsize=(11,3))
        plt.plot(range(len(yte)), yte, 'o-', label='Actual', ms=3, lw=1)
        plt.plot(range(len(yte)), yp, 's--', label='Prediksi', ms=3, lw=1)
        plt.title(f'{nama}  (R2={hasil["R2"]}, MAE={hasil["MAE"]}, MAPE={hasil["MAPE"]}%)
(CI 95%: [{hasil["Batas_bawah_95"]}, {hasil["Batas_atas_95"]}] | sigma_val: {sigma:.4f})')
        plt.legend(); plt.tight_layout(); plt.show()
    return hasil, model


## 7. Jalankan untuk SEMUA Parameter Numerik (12 model terpisah)

In [ ]:
hasil_semua, models = [], {}
for nama, col in numeric_params.items():
    h, m = proses_parameter(nama, df[col], plot=True)
    hasil_semua.append(h); models[nama] = m

ringkasan = pd.DataFrame(hasil_semua)
print('\n================= RINGKASAN SEMUA PARAMETER =================')
print(ringkasan.to_string(index=False))

## 8. Tabel Prediksi Nilai Berikutnya (+ interval 95%) & Simpan

In [ ]:
tabel = ringkasan[['Parameter','Prediksi_berikutnya','Batas_bawah_95','Batas_atas_95','R2','MAPE']].copy()
# tambahkan baris Appearance
ape_val = df['APE'].astype(str).str.strip()
ape_val = ape_val[ape_val.str.lower()!='nan'].mode().iloc[0]
tabel.loc[len(tabel)] = ['Appearance', ape_val, '-', '-', '-', '-']
print(tabel.to_string(index=False))
ringkasan.to_csv('hasil_prediksi_univariat.csv', index=False)
print('\nDisimpan: hasil_prediksi_univariat.csv')

## 9. Kesimpulan & Interpretasi  **(jujur)**

- **Cara baca R²:** R² mendekati 0 = model hanya sebaik menebak rata-rata; R² negatif = lebih buruk dari rata-rata.
- Pada data ini, R² tiap parameter umumnya **mendekati 0**, menandakan nilai antar-batch **berfluktuasi acak** dan **sulit diprediksi dari riwayatnya sendiri** (autokorelasi antar batch ≈ 0). Temuan serupa dilaporkan ketika data berpola stabil/acak: model deret waktu sederhana dapat menyamai atau mengungguli ML (lihat referensi domain di bawah).
- Walau model sering **mengalahkan baseline naif**, itu karena baseline naif sangat buruk pada data acak — jadi **"menang" ≠ "akurat"**.
- **Implikasi praktis:** untuk prediksi per batch yang akurat, riwayat parameter saja tidak cukup; diperlukan **parameter proses produksi** (suhu, waktu reaksi, bahan baku) atau prediksi pada level **agregat (mingguan/bulanan)** yang lebih stabil.

## Daftar Pustaka

**Metode & Pembelajaran Mesin**

1. Box, G. E. P., & Jenkins, G. M. (1970). *Time Series Analysis: Forecasting and Control.* Holden-Day.
2. Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785-794. https://doi.org/10.1145/2939672.2939785
3. Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts. https://otexts.com/fpp3/
4. Hyndman, R. J., & Koehler, A. B. (2006). Another look at measures of forecast accuracy. *International Journal of Forecasting, 22*(4), 679-688. https://doi.org/10.1016/j.ijforecast.2006.03.001
5. Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: an analysis and review. *International Journal of Forecasting, 16*(4), 437-450. https://doi.org/10.1016/S0169-2070(00)00065-0

**Penerapan XGBoost / Prediksi Produksi & Kualitas (Jurnal Indonesia)**

6. Winurputra, R., & Ratnawati, D. E. (2025). Peramalan Penjualan Produk Menggunakan Extreme Gradient Boosting (XGBoost) dan Kerangka Kerja CRISP-DM untuk Pengoptimalan Manajemen Persediaan (Studi Kasus: UB Mart). *Jurnal Teknologi Informasi dan Ilmu Komputer (JTIIK), 12*(2), 417-428. https://doi.org/10.25126/jtiik.2025129451
7. Darusman, Abbas, A., & Firmanto, A. D. (2025). Prediksi Kualitas Produk Manufaktur Semikonduktor Menggunakan Machine Learning. *Informatics for Educators and Professional (ITBI), 10*(1). https://doi.org/10.51211/itbi.v10i1.3348
8. Azizah, N. N. (2026). Algoritma XGBoost untuk Analisis Prediksi Permintaan Kalibrasi di PT PAL Indonesia. *JIKO (Jurnal Informatika dan Komputer)*, Universitas Teknologi Digital Indonesia. https://ejournal.akakom.ac.id/index.php/jiko/article/view/2245
9. *Model Prediksi Produksi Pertanian Berbasis Machine Learning dan Data Lapangan.* (2025). Jurnal SISFO, Universitas Malikussaleh. https://ojs.unimal.ac.id/sisfo/article/view/26015
